# Centauri Bloom — final 14M language experiment
Kaggle GPU · 1,500 steps · 25M TinyStories source tokens. Continue the latest canonical 13,989,888-parameter checkpoint, with no instruction tuning. Keep the notebook private while using secrets. Enable Internet and an NVIDIA GPU; one GPU is sufficient. Review [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) (CDLA-Sharing-1.0) and checkpoint terms, then confirm permission below. No Kaggle dataset attachment is required. Optional HF_TOKEN Kaggle secret for private access/PR upload; public downloads work without one if the repository is public. This experiment does not establish that 14M is a fundamental capacity limit.

In [ ]:
CONFIRM_DATA_PERMISSION = False  # Set True after reviewing source terms.
STEPS = 1500
TRAIN_BUDGET = 25_000_000
VAL_BUDGET = 500_000
DATA_SEED = 20261126
CODE_REV = "4216094a9639fee1dacca28f9ed452d93b7752bb"
if not CONFIRM_DATA_PERMISSION:
    raise ValueError("Review TinyStories terms and set CONFIRM_DATA_PERMISSION = True.")

In [ ]:
import os, sys, subprocess, json, hashlib, shutil, time
from pathlib import Path
from types import SimpleNamespace
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable an NVIDIA GPU in Kaggle settings.")
ROOT = Path('/kaggle/working')
if not ROOT.is_dir():
    raise RuntimeError("This notebook targets Kaggle.")
REPO = ROOT / 'centauri-bloom'
os.environ['HF_HUB_DISABLE_XET'] = '1'
def run(*args):
    subprocess.run(list(args), check=True)
if not (REPO/'.git').exists():
    run('git','clone','https://github.com/InterstellarNerd8760/centauri-bloom.git',str(REPO))
run('git','-C',str(REPO),'checkout','--detach',CODE_REV)
run(sys.executable,'-m','pip','install','--no-deps','-e',str(REPO))
run(sys.executable,'-m','pip','install','tokenizers','datasets','huggingface_hub','safetensors','gguf')
sys.path.insert(0,str(REPO))
from bloom import trainer
from bloom.trainer_data import encode_windows
from bloom.notebook_workflow import preflight, submit_candidate, sha256
from bloom.gguf_export import export_candidate
from huggingface_hub import HfApi, hf_hub_download
from datasets import load_dataset
from torch.utils.data import DataLoader
preflight('cuda')
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    token = None
api = HfApi(token=token or False)
del token
model_info = api.model_info(trainer.REPO_ID)
revision = model_info.sha
assert revision, 'Could not pin parent revision'
parent_metrics = json.loads(Path(hf_hub_download(trainer.REPO_ID,'metrics.json',revision=revision,token=api.token)).read_text())
assert parent_metrics['parameters'] == 13_989_888, 'Parent is no longer the 14M model'
pinned_api = SimpleNamespace(token=api.token, model_info=lambda _: model_info)
OUT = ROOT / ('bloom-14m-final-' + time.strftime('%Y%m%d-%H%M%S',time.gmtime()))
print('GPU:',torch.cuda.get_device_name(0),'Parent:',revision,'Step:',parent_metrics['step'],'Output:',OUT)

## Dataset
Use the official TinyStories validation split, kept separate from training. Skip the first 200,000 training records before deterministic shuffle to reduce overlap with earlier bounded streaming runs. Record dataset revision and hashes of selected token sequences. Earlier runs may still overlap; this is not a verified globally unseen corpus. Exactly duplicate token sequences are excluded within and across the two splits. The original tokenizer and model architecture are preserved.

In [ ]:
DATA_REPO = 'roneneldan/TinyStories'
RECIPE = 'TinyStories final 14M language experiment'
def prepare_final(recipe, repo, tok, context, eot, seed, jaron_path=None):
    dataset_rev = HfApi().dataset_info(DATA_REPO).sha
    assert dataset_rev, 'Dataset revision unavailable'
    seen = set()
    records = {}
    def collect(split, budget):
        stream = load_dataset(DATA_REPO, split=split, revision=dataset_rev, streaming=True)
        if split == 'train':
            stream = stream.skip(200_000).shuffle(seed=DATA_SEED,buffer_size=10_000)
        docs, hashes, total = [], [], 0
        for row in stream:
            text = row.get('text')
            if not isinstance(text,str) or not text.strip():
                continue
            ids = tok.encode(text.strip()).ids[:4096]
            if len(ids)<16:
                continue
            key = hashlib.sha256(json.dumps(ids,separators=(',',':')).encode()).hexdigest()
            if key in seen:
                continue
            seen.add(key); hashes.append(key); docs.append(ids); total += len(ids)+1
            if total >= budget:
                break
        if total < budget:
            raise RuntimeError(f'{split}: only {total:,} tokens; training not started')
        records[split] = dict(tokens=total,documents=len(docs),token_sequence_sha256=hashes)
        print(split, f'{total:,} source tokens',len(docs),'documents',flush=True)
        return encode_windows(docs,context,eot)
    val = collect('validation',VAL_BUDGET)
    train = collect('train',TRAIN_BUDGET)
    meta = dict(recipe=recipe,training_track='continued-pretraining',dataset=DATA_REPO,revision=dataset_rev,license='CDLA-Sharing-1.0',seed=DATA_SEED,train_skip=200_000,shuffle_buffer=10_000,records=records,note='Official validation split; prior-run overlap not ruled out.')
    return train,val,{'TinyStories official validation':val},meta
trainer.RECIPES[RECIPE] = ('continued-pretraining',('TinyStories',))
trainer.prepare = prepare_final

## Fixed benchmark
The twelve legacy prompts below combine the previous notebook's text-completion and instruction prompts with the original unified trainer prompts (duplicates removed). Fifty new prompts are authored for this experiment: 40 language continuations and 10 question-answering/instruction regression checks. They are not added to training. Prior corpus coincidence cannot be excluded. The exact prompt strings and completions are saved for before/after review. Generation is greedy, up to 64 new tokens, stopping at EOS. Historical output lengths may differ, so rerun both sides. No automatic claim of intelligence is made.

In [ ]:
LEGACY = ['Hello','The Earth orbits','Photosynthesis is','Once upon a time','A computer program is','User: What is 2 plus 2?\nAssistant:','User: Hello, how are you?\nAssistant:','User: What is Earth?\nAssistant:','User: Explain why the sky is blue.\nAssistant:','User: What is 12 times 5?\nAssistant:','User: Write a Python function that adds two numbers.\nAssistant:','User: Write a short story about a robot.\nAssistant:']
LANGUAGE = [
 'Mira found a silver button under the library table.','The little dog waited by the garden gate because','After the rain stopped, the children','A yellow umbrella lay beside the old bridge.','The baker forgot to close the window, so','When the lantern went out, Leo','Nora carried a paper boat to the pond.','The cat climbed onto the shelf and','A tiny bird landed on the fence.','The train was late, but the passengers','Inside the blue box, there was','The rabbit wanted to share its carrots with','On Saturday morning, the family','The bell rang just as the teacher','Sam was worried about his missing shoe.','The wind blew the red kite into','Two friends decided to plant a tree.','The mouse heard a quiet sound behind','When the snow melted, a small','The girl made a birthday card for','A robot named Pip found a broken clock.','The boat drifted toward the shore until','The farmer opened the barn door and','A little fox tried to cross the stream.','The boy apologized because he had','The shopkeeper helped a visitor find','The turtle was slower than the others, but','The cloud looked like a giant fish.','The children followed a trail of leaves.','A green scarf was hanging from','The birds began singing when','The suitcase was too heavy for','The moon shone through the bedroom window.','At the picnic, everyone brought','A child dropped a toy and then','The music stopped suddenly, and','The little bear could not find its friend.','Ravi put the wet clothes near','The old bicycle needed a new wheel.','Before going home, the friends decided to']
QUESTIONS = ['What color is grass?','Name a fruit that grows on a tree.','What is 3 plus 4?','How many legs does a dog have?','What do plants need to grow?','What is the opposite of cold?','Write a sentence about a blue bird.','Why do people use umbrellas?','Say hello politely.','What is the capital of France?']
PROMPTS = LEGACY + LANGUAGE + ['User: '+q+'\nAssistant:' for q in QUESTIONS]
assert len(LANGUAGE)==40 and len(QUESTIONS)==10 and len(set(PROMPTS))==len(PROMPTS)
BENCHMARK_DETAILS=[]
@torch.no_grad()
def benchmark(model,tok,context,eot,device):
    model.eval()
    results={}
    details={}
    for prompt in PROMPTS:
        ids=tok.encode(prompt).ids or [eot]
        new=[]; stopped=False
        for _ in range(64):
            x=torch.tensor([ids[-context:]],device=device,dtype=torch.long)
            nxt=int(model(x)[0,-1].argmax())
            if nxt==eot:
                stopped=True; break
            ids.append(nxt); new.append(nxt)
        grams=[tuple(new[i:i+4]) for i in range(max(0,len(new)-3))]
        results[prompt]=tok.decode(new)
        details[prompt]=dict(generated_tokens=len(new),eos=stopped,repeated_4gram_fraction=1-len(set(grams))/len(grams) if grams else 0.0)
    BENCHMARK_DETAILS.append(details)
    return results
trainer.generate_samples = benchmark

## Train and save
Uses the canonical trainer's causal loss, 16 sequences per microbatch, 2 accumulated microbatches, context 512, fresh AdamW, learning rate 5e-5, warmup and cosine decay. At 1,500 attempted steps it processes 24,576,000 token positions, roughly one pass. Saves safetensors every 500 steps and a final checkpoint, config, tokenizer, provenance and evaluation. Run this cell once. If interrupted, do not overwrite existing output; optimizer resume is not provided.

In [ ]:
metrics = trainer.run_training(pinned_api,recipe=RECIPE,steps=STEPS,repo_dir=REPO,out_dir=OUT)
assert metrics['parameters']==13_989_888
(OUT/'benchmark-generation-details.json').write_text(json.dumps(dict(parent=BENCHMARK_DETAILS[0],candidate=BENCHMARK_DETAILS[1]),indent=2))
report = json.loads((OUT/'evaluation.json').read_text())
print('Official validation loss:',report['parent_source_val_losses'],'->',report['candidate_source_val_losses'])
print('Successful optimizer updates:',metrics['optimizer_updates'],'/',STEPS)
# Record exact notebook experiment settings without changing hashed evaluation or provenance.
(OUT/'experiment.json').write_text(json.dumps(dict(code_revision=CODE_REV,train_budget=TRAIN_BUDGET,val_budget=VAL_BUDGET,legacy_prompts=LEGACY,language_prompts=LANGUAGE,regression_questions=QUESTIONS,generation_max_tokens=64,decision='Manual language coherence comparison; no automatic capacity claim.'),indent=2))

## Compare and decide
Review the 40 new language continuations using a 0–2 rubric: 0 = gibberish, empty or unrelated; 1 = understandable but seriously repetitive, incoherent or ungrammatical; 2 = relevant, mostly grammatical and coherent. Keep question-answering separate. Suggested threshold: at least 28/40 language outputs score 2, or at least 8 more score 2 than the parent with improved held-out loss. Otherwise move to a fresh ~50M architecture, carrying forward the benchmark and data discipline. This is a project decision threshold, not proof of a capacity limit.

In [ ]:
import csv
from IPython.display import display, FileLink
before,after=report['parent_samples'],report['candidate_samples']
with (OUT/'benchmark-comparison.csv').open('w',newline='',encoding='utf-8') as f:
    w=csv.writer(f); w.writerow(['category','prompt','parent','candidate','parent_score','candidate_score'])
    for p in PROMPTS:
        category='legacy' if p in LEGACY else 'language' if p in LANGUAGE else 'qa_regression'
        w.writerow([category,p,before[p],after[p],'',''])
for p in LANGUAGE[:8]:
    print(p,'\n BEFORE:',before[p],'\n AFTER: ',after[p],'\n')
print('Full comparison:',OUT/'benchmark-comparison.csv')

## Export and optional Hugging Face candidate PR
Keep the safetensors even if GGUF conversion or upload fails. GGUF is an experimental export; runtime parity is not established by a tensor round-trip. The PR is optional and never automatically merges. Set SUBMIT_HF_PR=True in the cell below only after reviewing the report and authorizing the HF_TOKEN secret. Download or save notebook outputs before ending the session.

In [ ]:
try:
    gguf_path=export_candidate(OUT,STEPS,metrics)
    print('GGUF:',gguf_path)
except Exception as exc:
    print('GGUF export failed; safetensors remain intact:',str(exc))
# Archive only model and reports, not the processed corpus or repository.
import zipfile
bundle=ROOT/(OUT.name+'.zip')
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_STORED) as z:
    for p in OUT.iterdir():
        if p.is_file() and p.suffix in {'.json','.csv','.safetensors','.gguf'}:
            z.write(p,p.name)
print('Bundle:',bundle)
display(FileLink(str(bundle.relative_to(ROOT))))

In [ ]:
SUBMIT_HF_PR = False  # Change only after reviewing results.
if SUBMIT_HF_PR:
    if not api.token:
        raise RuntimeError('Enable an authorized HF_TOKEN Kaggle secret and rerun setup or authenticate separately.')
    submit_candidate(api,trainer.REPO_ID,OUT,STEPS,revision,metrics['run_name'])
else:
    print('No upload requested. Save/download the bundle and review results first.')